## Imports

In [1]:
import math
import os
import time
import requests
import pandas as pd
import json
import numpy as np
from pathlib import Path
from typing import Optional, Dict, List, Union
from unidecode import unidecode
import matplotlib.pyplot as plt

import pandas_gbq
from google.auth import default
from google.cloud import bigquery
from google.api_core.exceptions import NotFound

import warnings
warnings.filterwarnings("ignore")

/Users/lucas.nascimento/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/lucas.nascimento/Library/Python/3.9/lib/python/site-packages/google/api_core/_python_version_support.py:242: FutureWarning: You are using a non-supported Python version (3.9.6). Google will not post any further updates to google.api_core supporting this Python version. Please upgrade to the latest Python version, or at least Python 3.10, and then update google.api_core.
  warnings.warn(message, FutureWarning)
/Users/lucas.nascimento/Library/Python/3.9/lib/python/site-packages/google/auth/__init__.py:54: FutureWarning: You are using a Python version 3.9 past its end of life. Google will update google-auth with critical bug fixes on a best-effort basis, but not with any other fixes or features. Please u

In [2]:
from funcoes_escoragem import *

## Diretório

In [3]:
BASE_DIR = Path("data")
RAW_DIR = BASE_DIR / "raw"
TRUSTED_DIR = BASE_DIR / "trusted"
ANALYTICS_DIR = BASE_DIR / "analytics"

for path in [RAW_DIR, TRUSTED_DIR, ANALYTICS_DIR]:
    path.mkdir(parents=True, exist_ok=True)

## Google BigQuery

In [4]:
project_id = 'loft-dl-fintech'

# CredPago - Consulta Realizada**

In [5]:
query_credpago = f"""
WITH consulta_realizada AS (
    SELECT
        CAST(REGEXP_REPLACE(documento, r'[^0-9]', '') AS INT64) AS CPF_CNPJ,
        id_externo AS contract_id,

        MIN(DATE(data)) OVER (
            PARTITION BY id_externo
        ) AS requested_at,

        MAX(DATE(data)) OVER (
            PARTITION BY id_externo
        ) AS data_ultima_consulta,
        ARRAY_LENGTH(JSON_EXTRACT_ARRAY(CR.json_retornado, '$.pessoas')) AS qtd_proponentes,
        CR.*
    FROM `loft-dl-fintech.bronze_credpago_sortinghat.consulta_realizada` CR
    WHERE DATE(data) >= DATE_SUB(CURRENT_DATE(), INTERVAL 4 WEEK)
    AND DATE(data) < CURRENT_DATE()
)

SELECT *
FROM consulta_realizada
QUALIFY ROW_NUMBER() OVER (
    PARTITION BY contract_id
    ORDER BY 
        CASE WHEN model = 'BLEND_4' THEN 1 ELSE 2 END ASC,
        data DESC
) = 1
"""

credpago_df = pd.read_gbq(query_credpago, project_id=project_id)
credpago_df

,CPF_CNPJ,contract_id,requested_at,data_ultima_consulta,qtd_proponentes,score_imobiliaria,renda_considerada,model,id_externo,json_retornado,...,id_funcionalidade,_sdc_table_version,request,_sdc_received_at,_sdc_sequence,documento,rating,_sdc_batched_at,data,result
0,12132066698,4193151,2026-06-26,2026-06-26,1,B,1781.000000000,BLEND3_3,4193151,"{""pessoas"":[{""nome"":""SAYONARA CRISTINA HORTA D...",...,32,1778785248777,"{""valor_aluguel"":950,""matchmaking_on"":false,""p...",2026-06-27 08:00:48+00:00,1782547246412981122,12132066698,B,2026-06-27 08:17:33.729000+00:00,2026-06-26 17:12:51+00:00,APROVAR
1,45083478862,4232302,2026-06-29,2026-06-29,1,E,2877.000000000,BLEND3_3,4232302,"{""pessoas"":[{""nome"":""ISABELLE DE SA MONDINI"",""...",...,34,1778785248777,"{""valor_aluguel"":950,""matchmaking_on"":false,""p...",2026-06-29 18:00:41+00:00,1782756037822259752,45083478862,C,2026-06-29 18:14:42.176000+00:00,2026-06-29 10:43:36+00:00,DERIVAR
2,78767946100,4251327,2026-06-28,2026-07-12,1,NI,4247.000000000,BLEND3_3,4251327,"{""pessoas"":[{""nome"":""KEILA RIBEIRO DE MELO"",""d...",...,34,1778785248777,"{""valor_aluguel"":1600,""matchmaking_on"":false,""...",2026-07-13 08:00:35+00:00,1783929632901290199,78767946100,C,2026-07-13 08:03:17.336000+00:00,2026-07-12 18:58:27+00:00,APROVAR
3,1668565099,4251748,2026-06-22,2026-06-22,1,NI,7124.000000000,BLEND3_3,4251748,"{""pessoas"":[{""nome"":""FRANCISCO CARDOSO SOARES""...",...,32,1778785248777,"{""valor_aluguel"":""1150"",""imobiliaria_id"":5801,...",2026-06-23 08:00:35+00:00,1782201632420304444,01668565099,B,2026-06-23 08:12:37.491000+00:00,2026-06-22 16:25:52+00:00,APROVAR
4,34225488829,4251858,2026-07-02,2026-07-02,1,C,4521.000000000,BLEND3_3,4251858,"{""pessoas"":[{""nome"":""GIULIA MARTINS"",""document...",...,32,1778785248777,"{""valor_aluguel"":1700,""matchmaking_on"":false,""...",2026-07-02 18:00:45+00:00,1783015239386587134,34225488829,B,2026-07-02 18:07:43.501000+00:00,2026-07-02 09:20:48+00:00,APROVAR
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
112761,53401027859,4379348,2026-07-15,2026-07-15,1,NI,0E-9,BLEND3_3,4379348,"{""pessoas"":[{""nome"":"""",""documento"":""5340102785...",...,33,1778785248777,"{""valor_aluguel"":1100,""matchmaking_on"":false,""...",2026-07-16 08:00:43+00:00,1784188841848917529,53401027859,E,2026-07-16 08:01:28.929000+00:00,2026-07-15 18:00:41+00:00,REPROVAR
112762,40804900892,4379463,2026-07-15,2026-07-15,1,NI,32743.000000000,BLEND3_3,4379463,"{""pessoas"":[{""nome"":""BRUNO LOURENCO PACHECO"",""...",...,32,1778785248777,"{""valor_aluguel"":2000,""matchmaking_on"":false,""...",2026-07-16 08:00:43+00:00,1784188842090269519,40804900892,B,2026-07-16 08:01:28.982000+00:00,2026-07-15 19:06:52+00:00,APROVAR
112763,51315391880,4379559,2026-07-15,2026-07-15,1,D,1918.000000000,BLEND3_3,4379559,"{""pessoas"":[{""nome"":""NAYARA AFONSO ARTZ DE SOU...",...,33,1778785248777,"{""valor_aluguel"":1700,""matchmaking_on"":false,""...",2026-07-16 08:00:43+00:00,1784188842262635953,51315391880,E,2026-07-16 08:01:29.022000+00:00,2026-07-15 21:02:41+00:00,REPROVAR
112764,50048141801,4379562,2026-07-15,2026-07-15,1,NI,1849.500000000,BLEND3_3,4379562,"{""pessoas"":[{""nome"":""THAILA FERREIRA ADORNO"",""...",...,33,1778785248777,"{""valor_aluguel"":1200,""matchmaking_on"":false,""...",2026-07-16 08:00:43+00:00,1784188842279366510,50048141801,E,2026-07-16 08:01:29.024000+00:00,2026-07-15 21:19:02+00:00,APROVAR


In [6]:
mask = credpago_df['contract_id'].astype(str).str.match(r'^\d+$')
credpago_df = credpago_df[mask].copy()
credpago_df['contract_id'] = credpago_df['contract_id'].astype(int)

In [7]:
credpago_df['model'].value_counts(dropna=False)

model
BLEND3_3                95230
BLEND_4                  5959
BLEND_REGRESSAO_2026     5120
HVA3                     4426
BVS_CUSTOM               1158
HVA4                      700
BVS_CUSTOM_V2             138
HFT1                       24
                           11
Name: count, dtype: int64

## Multiproponente

In [8]:
credpago_df['qtd_proponentes'].value_counts(dropna=False)

qtd_proponentes
1    108401
2      4000
3       323
4        37
5         3
8         1
6         1
Name: count, dtype: Int64

In [9]:
credpago_df['qtd_proponentes'].value_counts(normalize=True, dropna=False)

qtd_proponentes
1    0.961292
2    0.035472
3    0.002864
4    0.000328
5    0.000027
8    0.000009
6    0.000009
Name: proportion, dtype: Float64

## Quebrar JSON - Retornado

In [10]:
def unwrap_payload(obj):
    """Supports old format (message wrapper) and new format (root payload)."""
    if not obj:
        return None
    if isinstance(obj, dict) and "message" in obj:
        return obj["message"]
    return obj

def get_pessoas(obj):
    payload = unwrap_payload(obj)
    return (payload or {}).get("pessoas") or []

In [11]:
parsed = credpago_df["json_retornado"].apply(
    lambda x: json.loads(x) if pd.notna(x) and x else None
)

valid_parsed = parsed.dropna()
payload_parsed = valid_parsed.apply(unwrap_payload)

# json_normalize resets the index; preserve credpago_df labels for the join below
contrato_df = pd.json_normalize(payload_parsed.tolist(), sep="_")
contrato_df.index = payload_parsed.index
contrato_df = contrato_df.add_prefix("message_")  # keeps message_decisao, message_scores_*, etc.

pessoa_records = payload_parsed.apply(lambda x: (get_pessoas(x) or [{}])[0])
pessoa_df = pd.json_normalize(pessoa_records.tolist(), sep="_")
pessoa_df.index = payload_parsed.index
pessoa_df = pessoa_df.add_prefix("pessoa_")

In [12]:
valid = parsed.notna()
base_idx = credpago_df.loc[valid].index

pendencias = []
for idx, row in parsed[valid].items():
    for p in get_pessoas(row):
        if not isinstance(p, dict):
            continue

        serasa = p.get("anotacoesFinanceirasSerasa") or {}
        pend_list = serasa.get("PendenciasPagamentoPF") or []

        for pend in pend_list:
            if isinstance(pend, dict):
                pendencias.append({"idx": idx, **pend})

pendencias_df = pd.DataFrame(pendencias)

if not pendencias_df.empty:
    pendencias_agg = (
        pendencias_df
        .groupby("idx", as_index=False)
        .agg(
            qtde_pefin=("Valor", "count"),
            valor_pefin_total=("Valor", lambda s: pd.to_numeric(s, errors="coerce").sum()),
            modalidades_pefin=("Modalidade", lambda s: ",".join(sorted(set(s.dropna().astype(str))))),
        )
    )
else:
    pendencias_agg = pd.DataFrame(columns=["idx", "qtde_pefin", "valor_pefin_total", "modalidades_pefin"])

## Quebrar JSON - Request


In [13]:
request_parsed = credpago_df["request"].apply(
    lambda x: json.loads(x) if pd.notna(x) and x else None
)

request_valid = request_parsed.dropna()
request_df = pd.json_normalize(request_valid.tolist(), sep="_")
request_df.index = request_valid.index
request_df = request_df.add_prefix("request_")

# Unify old (snake_case) and new (camelCase) request schemas
REQUEST_FIELD_ALIASES = {
    "request_valorAluguel": ["request_valor_aluguel"],
    "request_imobiliariaId": ["request_imobiliaria_id"],
    "request_idExterno": ["request_id_externo"],
    "request_imovelTipo": ["request_imovel_tipo"],
    "request_principalDocument": ["request_pessoa_principal_documento"],
}

for target, sources in REQUEST_FIELD_ALIASES.items():
    existing_sources = [c for c in sources if c in request_df.columns]
    if not existing_sources and target not in request_df.columns:
        continue
    if target not in request_df.columns:
        request_df[target] = pd.NA
    for source in existing_sources:
        if target == "request_valorAluguel":
            request_df[target] = request_df[target].combine_first(
                pd.to_numeric(request_df[source], errors="coerce")
            )
        else:
            request_df[target] = request_df[target].combine_first(request_df[source])
        request_df = request_df.drop(columns=[source])


## Join Json's

In [14]:
EXPANDED_PREFIXES = ("message_", "pessoa_", "request_")
expanded_cols = [c for c in credpago_df.columns if c.startswith(EXPANDED_PREFIXES)]

base_df = credpago_df.loc[valid].drop(columns=expanded_cols, errors="ignore")

credpago_expandido = (
    base_df
    .join(contrato_df, how="left")
    .join(pessoa_df, how="left")
    .join(request_df, how="left")
    .reset_index(names="idx")
    .merge(pendencias_agg, on="idx", how="left")
    .drop(columns="idx")
)


In [15]:
cols_drop = [
    # ingestão
    "_sdc_table_version", "_sdc_received_at", "_sdc_sequence", "_sdc_batched_at",

    # json bruto / containers
    "json_retornado",
    "request",
    "message_pessoas", "message_scores", "message_ratings",
    "pessoa_scores", "pessoa_ratings", "message_blend3Variables",
    "request_priorDecisions", "request_dadosAusentes", "request_errosTecnicos",
    "request_outras_pessoas", "request_blend3Variables", "request_scores",

    # bronze redundante
    "score_bvs",
    "renda_considerada",
    "decisao_bureau",
    "limite_mensal",

    # prováveis duplicatas
    "id_externo",
    "data",
    "request_month",

    # operacional / baixo valor (old format only — safe to keep)
    "success", "message_manual", "message_node",
    "message_reaproveitamentoConsultaMotor", "message_savingBureauPowerCurve",
    "message_logs", "message_partnerIds",
    "message_errosTecnicos", "message_dadosAusentes",
    "pessoa_errosTecnicos", "pessoa_dadosAusentes",
    "message_rentGuaranteeConstraints_rent_coverage_multiplier",
    "message_rentGuaranteeConstraints_exit_cost_multiplier",
    "message_rentGuaranteeConstraints_commission_percent",
    "message_rentGuaranteeConstraints_version",
]

credpago_clean = credpago_expandido.drop(columns=[c for c in cols_drop if c in credpago_expandido.columns])


In [16]:
credpago_clean[
    (pd.to_datetime(credpago_clean["requested_at"]).dt.normalize() == "2026-07-03")
    & (credpago_clean["message_decisao"] == "BLEND_4")
][["message_blend3Variables_realEstatePc4HistoryOver100Contracts", "message_blend3Variables_cityPc4HistoryOver100Contracts"]]

,message_blend3Variables_realEstatePc4HistoryOver100Contracts,message_blend3Variables_cityPc4HistoryOver100Contracts
619,0.000000,0.172243
1661,NaN,0.124402
1666,0.107527,0.114391
1698,0.071661,0.076923
1701,0.000000,0.000000
...,...,...
110026,0.000000,0.130864
110027,0.000000,0.000000
111158,0.121212,0.106942
111178,0.000000,0.076923


## Escoragem Blend4

In [17]:
credpago_clean_rep = credpago_clean.replace({
    "request_imovelTipo": {"RESIDENCIAL": 1, "COMERCIAL": 0}, 
    "message_blend3Variables_hasPreviousContracts": {True: 1, False: 0},
    "message_blend3Variables_hadOverdueInvoiceInPreviousContracts": {True: 1, False: 0}
}
)

In [18]:
rename_dict = {
    "message_scores_BVS_CUSTOM_V2": "score_proposto__bvs",#
    "message_scores_HFT1": "SERASA_CHSV5",
    "pessoa_idade": "age",
    "request_imovelTipo": "property_type",
    "message_qtdeRestricoesContrato": "qtde_restricoes__consulta_realizada",
    "request_valorAluguel": "rental_value",
    "message_rendaConsideradaContrato": "income",
    "message_blend3Variables_realEstatePc4HistoryOver100Contracts": "agency_pc4_mais_100_contratos__pc_categorias",
    "message_blend3Variables_cityPc4HistoryOver100Contracts": "city_pc4_mais_100_contratos__pc_categorias",
    "message_blend3Variables_hasPreviousContracts": "flag_tem__contratos_anteriores",
    "message_blend3Variables_hadOverdueInvoiceInPreviousContracts": "flag_teve_boleto_atrasado__contratos_anteriores",
}

In [19]:
vars_blend4 = ['score_proposto__bvs', 'SERASA_CHSV5', 'age', 'property_type', 'qtde_restricoes__consulta_realizada', 'rental_value', 'income', 'agency_pc4_mais_100_contratos__pc_categorias', 'city_pc4_mais_100_contratos__pc_categorias', 'flag_tem__contratos_anteriores', 'flag_teve_boleto_atrasado__contratos_anteriores']

id_vars = ['contract_id', 'requested_at', 'CPF_CNPJ', 'message_decisao', 'message_blendRegressaoPredict']

In [20]:
df_predict = (credpago_clean_rep.copy()).rename(columns=rename_dict)
df_predict["REGRA_BLEND_4"] = np.where(
    df_predict["score_proposto__bvs"] <= 334,
    "E_BVS",
    "BLEND4",
)

df_predict["SCORE_BVS"] = df_predict["score_proposto__bvs"]
df_predict["SCORE_SERASA"] = df_predict["SERASA_CHSV5"]
df_predict["RENDA"] = df_predict["income"]

df_predict.head()

,CPF_CNPJ,contract_id,requested_at,data_ultima_consulta,qtd_proponentes,score_imobiliaria,model,id_consulta,id_funcionalidade,documento,...,request_blend3Variables_hadOverdueInvoiceInPreviousContracts,request_blend3Variables_cityPc4HistoryOver100Contracts,request_blend3Variables_realEstatePc4HistoryOver100Contracts,qtde_pefin,valor_pefin_total,modalidades_pefin,REGRA_BLEND_4,SCORE_BVS,SCORE_SERASA,RENDA
0,12132066698,4193151,2026-06-26,2026-06-26,1,B,BLEND3_3,6011988,32,12132066698,...,NaN,NaN,NaN,2.0,716.0,"CRED CARTAO,ENERGIA ELET",BLEND4,NaN,NaN,1781.0
1,45083478862,4232302,2026-06-29,2026-06-29,1,E,BLEND3_3,6016062,34,45083478862,...,NaN,NaN,NaN,4.0,10978.0,"CRED CARTAO,FAT AGUA",BLEND4,NaN,NaN,2877.0
2,78767946100,4251327,2026-06-28,2026-07-12,1,NI,BLEND3_3,6082632,34,78767946100,...,NaN,NaN,NaN,NaN,NaN,NaN,BLEND4,NaN,NaN,4247.0
3,1668565099,4251748,2026-06-22,2026-06-22,1,NI,BLEND3_3,5985689,32,01668565099,...,NaN,NaN,NaN,NaN,NaN,NaN,BLEND4,NaN,NaN,7124.0
4,34225488829,4251858,2026-07-02,2026-07-02,1,C,BLEND3_3,6034019,32,34225488829,...,NaN,NaN,NaN,NaN,NaN,NaN,BLEND4,NaN,NaN,4521.0


In [21]:
df_predict.groupby("REGRA_BLEND_4", dropna=False).size()

REGRA_BLEND_4
BLEND4    112145
E_BVS        621
dtype: int64

In [22]:
# df_predict.to_csv(ANALYTICS_DIR/"df_predict_blend4_pre.csv", index=False)

## Criacão das Variáveis

In [23]:
df_predict_vars = prepare_blend4_variables(df_predict)
df_predict_escorada = predict_blend4_1(df_predict_vars)

## Criar Rating Blend4

In [24]:
score = pd.to_numeric(df_predict_escorada["pred_blend4_1_to_score"], errors="coerce")

conditions = [
    score.between(480, 1000),
    score.between(443, 479),
    score.between(408, 442),
    score.between(343, 407),
    score.between(0, 342),
]

choices = ["A", "B", "C", "D", "E"]

df_predict_escorada["rating_manual_blend4"] = np.select(conditions, choices, default=None)

In [25]:
score = pd.to_numeric(df_predict_escorada["message_blendRegressaoPredict"], errors="coerce")

conditions = [
    score.between(480, 1000),
    score.between(443, 479),
    score.between(408, 442),
    score.between(343, 407),
    score.between(0, 342),
]

choices = ["A", "B", "C", "D", "E"]

df_predict_escorada["rating_json_blend4"] = np.select(conditions, choices, default=None)

## Salvar Base como se fosse Blend4

In [26]:
# df_predict_blend4 = df_predict_escorada[df_predict_escorada["message_decisao"] == "BLEND3_3"].copy()
# df_predict_blend4["message_decisao"] = df_predict_blend4["message_decisao"].replace("BLEND3_3", "BLEND4")

# cp_final_df = pd.concat([df_predict_escorada, df_predict_blend4])
cp_final_df = df_predict_escorada.copy()
cp_final_df

,CPF_CNPJ,contract_id,requested_at,data_ultima_consulta,qtd_proponentes,score_imobiliaria,model,id_consulta,id_funcionalidade,documento,...,SERASA_CHSV5__normalized4_1,age__normalized4_1,qtde_restricoes__consulta_realizada__normalized4_1,income_commitment__normalized4_1,agency_pc4_mais_100_contratos__pc_categorias__normalized4_1,city_pc4_mais_100_contratos__pc_categorias__normalized4_1,pred_blend4_1,pred_blend4_1_to_score,rating_manual_blend4,rating_json_blend4
0,12132066698,4193151,2026-06-26,2026-06-26,1,B,BLEND3_3,6011988,32,12132066698,...,0.0,-0.10,2.0,0.228540,0.000000,-1.459541,0.480282,520.0,A,A
1,45083478862,4232302,2026-06-29,2026-06-29,1,E,BLEND3_3,6016062,34,45083478862,...,0.0,-0.30,4.0,-0.220315,0.000000,-0.728183,0.510545,489.0,A,A
2,78767946100,4251327,2026-06-28,2026-07-12,1,NI,BLEND3_3,6082632,34,78767946100,...,0.0,0.75,0.0,-0.117532,0.000000,1.867673,0.572721,427.0,C,C
3,1668565099,4251748,2026-06-22,2026-06-22,1,NI,BLEND3_3,5985689,32,01668565099,...,0.0,0.35,4.0,-0.593130,0.000000,-0.424435,0.511998,488.0,A,A
4,34225488829,4251858,2026-07-02,2026-07-02,1,C,BLEND3_3,6034019,32,34225488829,...,0.0,-0.50,0.0,-0.119108,0.000000,-0.811805,0.480289,520.0,A,A
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
112761,53401027859,4379348,2026-07-15,2026-07-15,1,NI,BLEND3_3,6104549,33,53401027859,...,0.0,0.00,0.0,0.000000,0.000000,0.161927,0.516777,483.0,A,None
112762,40804900892,4379463,2026-07-15,2026-07-15,1,NI,BLEND3_3,6104694,32,40804900892,...,0.0,-0.15,0.0,-0.814780,0.000000,-0.054047,0.490373,510.0,A,A
112763,51315391880,4379559,2026-07-15,2026-07-15,1,D,BLEND3_3,6104806,33,51315391880,...,0.0,-0.35,2.0,1.008130,-0.071058,-0.507195,0.565737,434.0,C,E
112764,50048141801,4379562,2026-07-15,2026-07-15,1,NI,BLEND3_3,6104813,33,50048141801,...,0.0,-0.40,5.0,0.483482,0.000000,-1.275805,0.540024,460.0,B,D


In [27]:
cp_final_df.groupby("message_decisao", dropna=False).size()

message_decisao
                           11
BLEND3_3                95230
BLEND_4                  5959
BLEND_REGRESSAO_2026     5120
BVS_CUSTOM               1158
BVS_CUSTOM_V2             138
HFT1                       24
HVA3                     4426
HVA4                      700
dtype: int64

In [28]:
bvs = pd.to_numeric(cp_final_df["SCORE_BVS"], errors="coerce")
score = pd.to_numeric(cp_final_df["pred_blend4_1_to_score"], errors="coerce")

conditions = [
    bvs <= 334,                         # corte customizado BVS → E
    score.between(763, 1000),           # 763 – 1000
    score.between(704, 762),            # 704 – 762
    score.between(653, 703),            # 653 – 703
    score.between(607, 652),            # 607 – 652
    score.between(562, 606),            # 562 – 606
    score.between(520, 561),            # 520 – 561
    score.between(480, 519),            # 480 – 519
    score.between(443, 479),            # 443 – 479
    score.between(408, 442),            # 408 – 442
    score.between(375, 407),            # 375 – 407
    score.between(343, 374),            # 343 – 374
    score.between(307, 342),            # 307 – 342
    score.between(0, 306),              # 0 – 306
]

choices = [
    "E.E",      # override BVS ≤ 334
    "1.A+",
    "2.A",
    "3.A",
    "4.B+",
    "5.B+",
    "6.B",
    "7.B",
    "8.C",
    "9.D+",
    "10.D",
    "11.D",
    "E-1.E",
    "E-2.E",
]

cp_final_df["rating_cl_pol_blend4"] = np.select(conditions, choices, default=None)

## Salvar

In [29]:
cp_final_df.groupby("REGRA_BLEND_4", dropna=False).size()

REGRA_BLEND_4
BLEND4    112145
E_BVS        621
dtype: int64

In [30]:
cp_final_df.groupby("message_decisao", dropna=False).size()

message_decisao
                           11
BLEND3_3                95230
BLEND_4                  5959
BLEND_REGRESSAO_2026     5120
BVS_CUSTOM               1158
BVS_CUSTOM_V2             138
HFT1                       24
HVA3                     4426
HVA4                      700
dtype: int64

In [31]:
cp_final_df.groupby("model", dropna=False).size()

model
                           11
BLEND3_3                95230
BLEND_4                  5959
BLEND_REGRESSAO_2026     5120
BVS_CUSTOM               1158
BVS_CUSTOM_V2             138
HFT1                       24
HVA3                     4426
HVA4                      700
dtype: int64

In [32]:
cp_final_df.groupby("REGRA_BLEND_4", dropna=False).size()

REGRA_BLEND_4
BLEND4    112145
E_BVS        621
dtype: int64

In [33]:
cp_final_df.to_csv(ANALYTICS_DIR/"df_predict_blend4.csv", index=False)